# Step 1: Data File Check

This notebook checks whether the Olist raw CSV files were correctly stored and can be loaded for the logistics optimization portfolio.

In [1]:
# Step 1: Check raw Olist data files for logistics optimization

from pathlib import Path
import platform
import sys

import pandas as pd
import numpy as np
import matplotlib
from IPython.display import display

# Set project directories
project_dir = Path("/Users/mac/Desktop/portfolio2_logistics_optimization")
data_raw_dir = project_dir / "data" / "raw"

# Print environment information
print("=== Environment Check ===")
print("Python executable:", sys.executable)
print("Python version:", platform.python_version())
print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)
print("matplotlib version:", matplotlib.__version__)

# List raw CSV files
print("\n=== Raw CSV Files ===")
csv_files = sorted(data_raw_dir.glob("*.csv"))

print("Number of CSV files:", len(csv_files))

for file in csv_files:
    print(file.name)

# Read each CSV file and summarize rows and columns
print("\n=== File Summary ===")

data_summary = []

for file in csv_files:
    df = pd.read_csv(file)
    
    data_summary.append({
        "file_name": file.name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

summary_df = pd.DataFrame(data_summary)

display(summary_df)

# Check whether required logistics variables exist
print("\n=== Required Logistics Variable Check ===")

required_columns = {
    "olist_orders_dataset.csv": [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp"
    ],
    "olist_order_items_dataset.csv": [
        "order_id",
        "seller_id",
        "price",
        "freight_value"
    ],
    "olist_customers_dataset.csv": [
        "customer_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ],
    "olist_sellers_dataset.csv": [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ],
    "olist_geolocation_dataset.csv": [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ]
}

check_results = []

for file_name, columns in required_columns.items():
    file_path = data_raw_dir / file_name
    
    if file_path.exists():
        df = pd.read_csv(file_path, nrows=5)
        existing_columns = set(df.columns)
        
        for column in columns:
            check_results.append({
                "file_name": file_name,
                "required_column": column,
                "exists": column in existing_columns
            })
    else:
        for column in columns:
            check_results.append({
                "file_name": file_name,
                "required_column": column,
                "exists": False
            })

check_df = pd.DataFrame(check_results)

display(check_df)

# Final result
all_required_columns_exist = check_df["exists"].all()

print("\n=== Step 1 Final Result ===")
print("All required logistics variables exist:", all_required_columns_exist)

if all_required_columns_exist:
    print("Step 1 is complete. The raw data files are ready for Portfolio 2.")
else:
    print("Step 1 has issues. Some required files or columns are missing.")
    display(check_df[check_df["exists"] == False])

=== Environment Check ===
Python executable: /opt/miniconda3/envs/ops311/bin/python
Python version: 3.11.15
pandas version: 3.0.3
numpy version: 2.4.6
matplotlib version: 3.10.9

=== Raw CSV Files ===
Number of CSV files: 9
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv

=== File Summary ===


,file_name,rows,columns
0,olist_customers_dataset.csv,99441,5
1,olist_geolocation_dataset.csv,1000163,5
2,olist_order_items_dataset.csv,112650,7
3,olist_order_payments_dataset.csv,103886,5
4,olist_order_reviews_dataset.csv,99224,7
5,olist_orders_dataset.csv,99441,8
6,olist_products_dataset.csv,32951,9
7,olist_sellers_dataset.csv,3095,4
8,product_category_name_translation.csv,71,2



=== Required Logistics Variable Check ===


,file_name,required_column,exists
0,olist_orders_dataset.csv,order_id,True
1,olist_orders_dataset.csv,customer_id,True
2,olist_orders_dataset.csv,order_status,True
3,olist_orders_dataset.csv,order_purchase_timestamp,True
4,olist_order_items_dataset.csv,order_id,True
5,olist_order_items_dataset.csv,seller_id,True
6,olist_order_items_dataset.csv,price,True
7,olist_order_items_dataset.csv,freight_value,True
8,olist_customers_dataset.csv,customer_id,True
9,olist_customers_dataset.csv,customer_zip_code_prefix,True



=== Step 1 Final Result ===
All required logistics variables exist: True
Step 1 is complete. The raw data files are ready for Portfolio 2.


# Step 2: Build Order-Customer-Seller Base Table

This step joins orders, order items, customers, and sellers to create the core logistics base table for assignment optimization.

In [2]:
# Step 2: Build order-customer-seller logistics base table

from pathlib import Path
import pandas as pd
from IPython.display import display

# Set project directories
project_dir = Path("/Users/mac/Desktop/portfolio2_logistics_optimization")
data_raw_dir = project_dir / "data" / "raw"
data_processed_dir = project_dir / "data" / "processed"

# Create processed data folder if it does not exist
data_processed_dir.mkdir(parents=True, exist_ok=True)

# Load core raw tables
orders = pd.read_csv(data_raw_dir / "olist_orders_dataset.csv")
order_items = pd.read_csv(data_raw_dir / "olist_order_items_dataset.csv")
customers = pd.read_csv(data_raw_dir / "olist_customers_dataset.csv")
sellers = pd.read_csv(data_raw_dir / "olist_sellers_dataset.csv")

# Print original table shapes
print("=== Original Table Shapes ===")
print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("customers:", customers.shape)
print("sellers:", sellers.shape)

# Check order status distribution
print("\n=== Order Status Distribution ===")
display(orders["order_status"].value_counts().reset_index().rename(
    columns={"order_status": "order_count", "index": "order_status"}
))

# Keep only delivered orders
delivered_orders = orders[orders["order_status"] == "delivered"].copy()

print("\n=== Delivered Orders ===")
print("delivered_orders:", delivered_orders.shape)

# Join delivered orders with order items
base_df = delivered_orders.merge(
    order_items,
    on="order_id",
    how="inner"
)

# Join customer information
base_df = base_df.merge(
    customers,
    on="customer_id",
    how="left"
)

# Join seller information
base_df = base_df.merge(
    sellers,
    on="seller_id",
    how="left"
)

# Select variables needed for logistics optimization
base_df = base_df[
    [
        "order_id",
        "customer_id",
        "seller_id",
        "order_purchase_timestamp",
        "order_status",
        "price",
        "freight_value",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ]
].copy()

# Convert order purchase timestamp to datetime
base_df["order_purchase_timestamp"] = pd.to_datetime(base_df["order_purchase_timestamp"])

# Check final base table shape
print("\n=== Logistics Base Table Shape ===")
print("order_logistics_base:", base_df.shape)

# Check missing values in key columns
print("\n=== Missing Values in Key Columns ===")
missing_summary = base_df.isna().sum().reset_index()
missing_summary.columns = ["column_name", "missing_count"]
missing_summary["missing_rate"] = missing_summary["missing_count"] / len(base_df)

display(missing_summary)

# Show sample rows
print("\n=== Sample Rows ===")
display(base_df.head())

# Save processed base table
output_path = data_processed_dir / "order_logistics_base.csv"
base_df.to_csv(output_path, index=False)

print("\n=== Step 2 Final Result ===")
print("Saved processed logistics base table to:")
print(output_path)

=== Original Table Shapes ===
orders: (99441, 8)
order_items: (112650, 7)
customers: (99441, 5)
sellers: (3095, 4)

=== Order Status Distribution ===


,order_count,count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2



=== Delivered Orders ===
delivered_orders: (96478, 8)

=== Logistics Base Table Shape ===
order_logistics_base: (110197, 13)

=== Missing Values in Key Columns ===


,column_name,missing_count,missing_rate
0,order_id,0,0.0
1,customer_id,0,0.0
2,seller_id,0,0.0
3,order_purchase_timestamp,0,0.0
4,order_status,0,0.0
5,price,0,0.0
6,freight_value,0,0.0
7,customer_zip_code_prefix,0,0.0
8,customer_city,0,0.0
9,customer_state,0,0.0



=== Sample Rows ===


,order_id,customer_id,seller_id,order_purchase_timestamp,order_status,price,freight_value,customer_zip_code_prefix,customer_city,customer_state,seller_zip_code_prefix,seller_city,seller_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-02 10:56:33,delivered,29.99,8.72,3149,sao paulo,SP,9350,maua,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,289cdb325fb7e7f891c38608bf9e0962,2018-07-24 20:41:37,delivered,118.70,22.76,47813,barreiras,BA,31570,belo horizonte,SP
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-08 08:38:49,delivered,159.90,19.22,75265,vianopolis,GO,14840,guariba,SP
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,66922902710d126a0e7d26b0e3805106,2017-11-18 19:28:06,delivered,45.00,27.20,59296,sao goncalo do amarante,RN,31842,belo horizonte,MG
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2c9e548be18521d1c43cde1c582c6de8,2018-02-13 21:18:39,delivered,19.90,8.72,9195,santo andre,SP,8752,mogi das cruzes,SP



=== Step 2 Final Result ===
Saved processed logistics base table to:
/Users/mac/Desktop/portfolio2_logistics_optimization/data/processed/order_logistics_base.csv
